# 8. RAG — Query Rewriter + Retriever + Generator

The app's LangGraph RAG concept decomposes RAG into three independently-testable
stages instead of one straight-line function: **query rewriter** (resolve
pronouns/references using conversation history — a pass-through if there's no
history yet), **retriever** (embedding similarity search), **generator**
(answer using the retrieved context). This notebook builds a simplified version
of all three, in-memory only — the real app additionally supports Cohere
reranking/eval and Redis/Postgres backends for both history and vectors (see
`docs/langgraph/08-rag.md`), left out here to keep the concept itself visible.

**Why a query rewriter at all:** if you ask "What is the capital of India?", get
"Delhi", then ask "Where is it located?", a literal search for "it" matches
nothing useful. The rewriter turns that into "Where is Delhi located?" using the
conversation so far.

**Prerequisites:** Ollama running locally with `llama3.2` and `qwen3-embedding:0.6b` pulled.

### Setup

This cell makes the project's shared `tools`/`models` packages importable
regardless of where Jupyter's working directory actually is (it's usually
this notebook's own folder, not the repo root), and loads `.env` plus any
cached secrets in `.env.local` (populated by `scripts/lib/env.sh` the first
time you've run `scripts/start_app.sh` / `scripts/start_infra.sh`).

In [ ]:
import sys
from pathlib import Path

from dotenv import load_dotenv

project_root = Path.cwd()
while not (project_root / "pyproject.toml").exists():
    project_root = project_root.parent
sys.path.insert(0, str(project_root))

load_dotenv(project_root / ".env")
load_dotenv(project_root / ".env.local", override=True)  # cached secrets, if resolve_env() has run at least once
print("Project root on sys.path:", project_root)

## Stage 1 — Query rewriter (pass-through if no history, else rewrite)

In [ ]:
from typing import TypedDict

from models.chat_models.ollama_models import SupportedModel, get_chat_model

llm = get_chat_model(SupportedModel.llama3_2)


class HistoryTurn(TypedDict):
    role: str
    content: str


def format_history(history: list[HistoryTurn]) -> str:
    return "\n".join(f"{turn['role']}: {turn['content']}" for turn in history)


def rewrite_query(question: str, history: list[HistoryTurn]) -> tuple[str, bool]:
    if not history:
        return question, False
    response = llm.invoke(
        f"Conversation so far:\n{format_history(history)}\n\n"
        f'The user\'s latest question is: "{question}"\n\n'
        "Rewrite the latest question as a standalone question that makes sense "
        "without the conversation above — resolve any pronouns or implicit "
        'references (e.g. "it", "that", "there") using the conversation. If the '
        "latest question is already standalone, return it unchanged. Respond "
        "with ONLY the rewritten question, nothing else."
    )
    return response.content.strip(), True


# Turn 1: no history yet -> pass-through.
print(rewrite_query("What is RAG used for?", []))

## Stage 2 — Retriever (ingest the sample corpus once, then similarity search)

In [ ]:
import uuid

from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

from models.embedding_models.ollama_models import qwen3_embedding_model

corpus_path = project_root / "langchain_demo" / "rag_corpus.md"
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_text(corpus_path.read_text())
documents = [Document(page_content=chunk, id=str(uuid.uuid5(uuid.NAMESPACE_URL, chunk))) for chunk in chunks]

vector_store = InMemoryVectorStore(embedding=qwen3_embedding_model)
vector_store.add_documents(documents)


def retrieve(query: str, k: int = 4) -> list[Document]:
    return vector_store.similarity_search(query, k=k)


print(f"ingested {len(documents)} chunks")

## Stage 3 — Generator (answer using retrieved context)

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

ANSWER_PROMPT = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "Answer the question using ONLY the provided context. If the context "
            "doesn't contain the answer, say so.\n\nContext:\n{context}",
        ),
        ("human", "{query}"),
    ]
)
generator_chain = ANSWER_PROMPT | llm


def generate(query: str, documents: list[Document]) -> str:
    context = "\n\n".join(doc.page_content for doc in documents)
    return generator_chain.invoke({"context": context, "query": query}).content

## Stitch the three stages together across a multi-turn conversation

In [ ]:
history: list[HistoryTurn] = []


def ask(question: str) -> str:
    rewritten, was_rewritten = rewrite_query(question, history)
    docs = retrieve(rewritten)
    answer = generate(rewritten, docs)
    history.append({"role": "human", "content": question})
    history.append({"role": "ai", "content": answer})
    print(f"question:   {question}")
    print(f"rewritten:  {rewritten}  (was_rewritten={was_rewritten})")
    print(f"answer:     {answer}")
    print()
    return answer


ask("What is RAG used for?")
ask("Which backend is best for that?")

## 🧪 Playground

**1. A third turn**, referencing the second turn's answer — confirm the rewriter keeps working across more than 2 turns.

In [ ]:
# TODO: ask() a third, referential question


**2. Reset `history = []`** and ask the SAME follow-up question on its own — confirm it now gets a pass-through and (likely) a worse-grounded answer, since "it"/"that" no longer resolves to anything.

In [ ]:
# TODO: history = []; ask('Which backend is best for that?') on its own


**3. Add reranking** — the real app uses Cohere's `rerank-v3.5` to re-order the top-k down to a smaller top-n before generation (needs `COHERE_API_KEY` in `.env`). Try wiring that in as a step between `retrieve` and `generate` if you have a key configured.

In [ ]:
# TODO: import cohere, rerank the retrieved docs, then call generate() with the reranked list
